# Retinal Fundus Feature Detection Experiments

## Dataset Selection
*TODO: Document why you hand-picked each of your 15 specific images here.*

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

def select_and_visualize_candidates(folder_path, num_samples=12):
    all_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif'))]
    samples = random.sample(all_files, min(num_samples, len(all_files)))
    cols = 4
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
    axes = axes.flatten()
    
    for i, file_name in enumerate(samples):
        img_path = os.path.join(folder_path, file_name)
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        axes[i].imshow(img)
        axes[i].set_title(file_name, fontsize=8)
        axes[i].axis('off')
        
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()

# --- Paths ---
base_dir = r"d:\AUIS\CV\CV retinal fundus project\usefull data"
normal_dir = os.path.join(base_dir, "0.0.Normal")
armd_dir = os.path.join(base_dir, "All ARMD images")
diabetic_dir = os.path.join(base_dir, r"Diabetic Retinopathy\train\images")

# --- Your Selections ---
normal_selected = [
    '1ffa9657-8d87-11e8-9daf-6045cb817f5b..JPG',
    '1ffa9656-8d87-11e8-9daf-6045cb817f5b..JPG',
    '1ffa9652-8d87-11e8-9daf-6045cb817f5b..JPG',
    '1ffa9655-8d87-11e8-9daf-6045cb817f5b..JPG',
    '1ffa9654-8d87-11e8-9daf-6045cb817f5b..JPG'
]

armd_selected = [
    '0_RFiMD_77_ARMD.png',
    '0_1kIM_24_ARMD.png',
    '0_aria_a_43_9.png',
    '0_RFiMD_17_ARMD.png',
    '0_ORID19_30_ARMD.png'
]

diabetic_selected = [
    'IMAGE_00361.jpg',
    'IMAGE_00248.jpg',
    'IMAGE_00124.jpg',
    'IMAGE_02210.jpg',
    'IMAGE_01927.jpg'
]


In [ ]:
# ==========================================
# Step 4: Preprocessing
# Here we run our Preprocessing test against ALL 15 of the images we curated.
# You will see the Original, basic Grayscale, Green Channel, and the final Green Channel with CLAHE.
# ==========================================

def test_preprocessing(image_path):
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load {image_path}. Did you put the real filename in the list?")
        return
        
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # 1. Standard Grayscale (For comparison)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # VARIATION A: Extract Green Channel (OpenCV is BGR, so index 1 is Green)
    b, green_channel, r = cv2.split(img)
    
    # VARIATION B: Apply CLAHE on the Green Channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_green = clahe.apply(green_channel)
    
    # Plotting
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original RGB")
    axes[0].axis('off')
    
    axes[1].imshow(gray, cmap='gray')
    axes[1].set_title("Standard Grayscale (Poor Contrast)")
    axes[1].axis('off')
    
    axes[2].imshow(green_channel, cmap='gray')
    axes[2].set_title("Variation A: Green Channel (Better Structure)")
    axes[2].axis('off')
    
    axes[3].imshow(clahe_green, cmap='gray')
    axes[3].set_title("Variation B: Green + CLAHE (Best Definition!)")
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()

# Combine all 15 curated image paths into one single mega-list list
all_15_image_paths = []
for name in normal_selected: all_15_image_paths.append(os.path.join(normal_dir, name))
for name in armd_selected:   all_15_image_paths.append(os.path.join(armd_dir, name))
for name in diabetic_selected: all_15_image_paths.append(os.path.join(diabetic_dir, name))

# Loop through and test preprocessing on EVERY single one of your 15 curated images.
for count, img_path in enumerate(all_15_image_paths, 1):
    print(f"[{count}/15] Applying Preprocessing on: {os.path.basename(img_path)}")
    test_preprocessing(img_path)


In [ ]:
# ==========================================
# Step 5: Edge Detection (Canny vs Sobel)
# Here we compare two edge detection algorithms:
# 1. Sobel: A basic mathematical gradient algorithm (thick, blurry edges).
# 2. Canny: An advanced version of Sobel that adds "Non-Maximum Suppression" to thin the messy Sobel edges down into beautiful single-pixel lines.
#
# We test Canny with 3 different parameter settings:
# - Sensitive (20/50): Low thresholds. Picks up noisy microscopic textures.
# - Balanced (40/100): Solid middle ground isolating blood vessels.
# - Strict (80/200): High thresholds. Only detects strongest, thickest primary vessels.
# ==========================================

def test_edge_detection(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load {image_path}")
        return
        
    b, green_channel, r = cv2.split(img)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    preprocessed = clahe.apply(green_channel)
    
    # Add slight Gaussian Blur to remove microscopic camera noise before edge detection
    blurred = cv2.GaussianBlur(preprocessed, (5, 5), 0)
    
    # --- 1. Sobel Edge Detection ---
    sobelx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_combined = cv2.magnitude(sobelx, sobely)
    sobel_combined = cv2.convertScaleAbs(sobel_combined)
    
    # --- 2. Canny Edge Detection (3 different parameter settings) ---
    canny_sensitive = cv2.Canny(blurred, threshold1=20, threshold2=50)
    canny_balanced  = cv2.Canny(blurred, threshold1=40, threshold2=100)
    canny_strict    = cv2.Canny(blurred, threshold1=80, threshold2=200)
    
    # Plotting
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    
    axes[0].imshow(preprocessed, cmap='gray')
    axes[0].set_title("Input (Green + CLAHE)")
    axes[0].axis('off')
    
    axes[1].imshow(sobel_combined, cmap='gray')
    axes[1].set_title("Sobel (Thick/Blurry Edges)")
    axes[1].axis('off')
    
    axes[2].imshow(canny_sensitive, cmap='gray')
    axes[2].set_title("Canny (Sensitive: 20/50)")
    axes[2].axis('off')
    
    axes[3].imshow(canny_balanced, cmap='gray')
    axes[3].set_title("Canny (Balanced: 40/100)")
    axes[3].axis('off')
    
    axes[4].imshow(canny_strict, cmap='gray')
    axes[4].set_title("Canny (Strict: 80/200)")
    axes[4].axis('off')
    
    plt.subplots_adjust(wspace=0.05)
    plt.show()

# Apply Edge Detection to ALL 15 images in sequence.
for count, img_path in enumerate(all_15_image_paths, 1):
    print(f"[{count}/15] Applying Edge Detection on: {os.path.basename(img_path)}")
    test_edge_detection(img_path)


In [ ]:
# ==========================================
# Step 5 - VARIATION B: Edge Detection (Canny vs LoG)
# Here we test Laplacian of Gaussian (LoG) against Canny.
# LoG applies a thick Gaussian blur and then takes the Laplacian (second derivative).
# It is famous for creating 'zero-crossings' which causes blood vessels to look somewhat like hollow blobs instead of continuous strokes.
# ==========================================

def test_edge_detection_log(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return
        
    b, green_channel, r = cv2.split(img)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    preprocessed = clahe.apply(green_channel)
    
    # Standard Gaussian Blur prior to detection
    blurred = cv2.GaussianBlur(preprocessed, (5, 5), 0)
    
    # --- 1. LoG (Laplacian of Gaussian) ---
    # The LoG is literally just calculating the Laplacian after applying a Gaussian Blur.
    laplacian = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    laplacian = cv2.convertScaleAbs(laplacian)
    
    # --- 2. Canny Edge Detection (3 different parameter settings) ---
    canny_sensitive = cv2.Canny(blurred, threshold1=20, threshold2=50)
    canny_balanced  = cv2.Canny(blurred, threshold1=40, threshold2=100)
    canny_strict    = cv2.Canny(blurred, threshold1=80, threshold2=200)
    
    # Plotting
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    
    axes[0].imshow(preprocessed, cmap='gray')
    axes[0].set_title("Input (Green + CLAHE)")
    axes[0].axis('off')
    
    axes[1].imshow(laplacian, cmap='gray')
    axes[1].set_title("LoG (Laplacian of Gaussian)")
    axes[1].axis('off')
    
    axes[2].imshow(canny_sensitive, cmap='gray')
    axes[2].set_title("Canny (Sensitive: 20/50)")
    axes[2].axis('off')
    
    axes[3].imshow(canny_balanced, cmap='gray')
    axes[3].set_title("Canny (Balanced: 40/100)")
    axes[3].axis('off')
    
    axes[4].imshow(canny_strict, cmap='gray')
    axes[4].set_title("Canny (Strict: 80/200)")
    axes[4].axis('off')
    
    plt.subplots_adjust(wspace=0.05)
    plt.show()

# Apply LoG vs Canny Edge Detection to ALL 15 images in sequence.
for count, img_path in enumerate(all_15_image_paths, 1):
    print(f"[{count}/15] Applying LoG Edge Detection on: {os.path.basename(img_path)}")
    test_edge_detection_log(img_path)


In [ ]:
# ==========================================
# Step 6: Corner Detection (Harris vs Shi-Tomasi)
# We are locating the intersections and bifurcations where blood vessels branch out.
# - Harris: The classic mathematically rigorous corner algorithm.
# - Shi-Tomasi: A direct mathematical improvement over Harris that rejects bad corners better.
# ==========================================
import numpy as np

def test_corner_detection(image_path):
    img = cv2.imread(image_path)
    if img is None: return
        
    b, green_channel, r = cv2.split(img)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    preprocessed = clahe.apply(green_channel)
    
    # We need a 3-channel color image so we can draw bright colored dots onto it!
    img_harris = cv2.cvtColor(preprocessed, cv2.COLOR_GRAY2BGR)
    img_shi = img_harris.copy()
    
    # --- 1. Harris Corner Detection ---
    # Harris strictly requires the image to be float32
    gray_float = np.float32(preprocessed)
    harris_dst = cv2.cornerHarris(gray_float, blockSize=2, ksize=3, k=0.04)
    
    # Dilate the result to make the dots slightly thicker and easier to see on the screen
    harris_dst = cv2.dilate(harris_dst, None)
    
    # Threshold the corners (only keep the strongest 1% of corners)
    # Draw RED dots directly onto the image matrix
    img_harris[harris_dst > 0.01 * harris_dst.max()] = [0, 0, 255]
    
    # --- 2. Shi-Tomasi Corner Detection ---
    # Parameters: maxCorners=1000, qualityLevel=0.01, minDistance=10
    corners = cv2.goodFeaturesToTrack(preprocessed, 1000, 0.01, 10)
    
    if corners is not None:
        corners = np.int32(corners)
        for i in corners:
            x, y = i.ravel()
            # Draw solid GREEN circles onto the image matrix
            cv2.circle(img_shi, (x, y), 5, (0, 255, 0), -1)
            
    # --- Plotting ---
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    axes[0].imshow(preprocessed, cmap='gray')
    axes[0].set_title("Input (Green + CLAHE)")
    axes[0].axis('off')
    
    # OpenCV uses BGR natively, Matplotlib uses RGB. Convert them before displaying!
    axes[1].imshow(cv2.cvtColor(img_harris, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Harris Corners (Red Dots)")
    axes[1].axis('off')
    
    axes[2].imshow(cv2.cvtColor(img_shi, cv2.COLOR_BGR2RGB))
    axes[2].set_title("Shi-Tomasi Corners (Green Dots)")
    axes[2].axis('off')
    
    plt.subplots_adjust(wspace=0.05)
    plt.show()

# Apply Corner Detection to ALL 15 images in sequence.
for count, img_path in enumerate(all_15_image_paths, 1):
    print(f"[{count}/15] Applying Corner Detection on: {os.path.basename(img_path)}")
    test_corner_detection(img_path)


In [ ]:
# ==========================================
# Step 7: Blob Detection (SIFT vs ORB)
# Here we calculate distinct regions of interest (blobs) across different scale-spaces.
# - SIFT: A classic, highly accurate DoG (Difference of Gaussians) based feature detector.
# - ORB: A modern, ultra-fast alternative to SIFT/SURF invented by OpenCV Labs.
#
# The colorful circles drawn automatically scale to the exact mathematical "size" of the Blob found!
# ==========================================

def test_blob_detection(image_path):
    img = cv2.imread(image_path)
    if img is None: return
        
    b, green_channel, r = cv2.split(img)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    preprocessed = clahe.apply(green_channel)
    
    # Base image for OpenCV to draw colorful rings onto
    base_color_img = cv2.cvtColor(preprocessed, cv2.COLOR_GRAY2BGR)
    
    # --- 1. SIFT Blob Detection (Difference of Gaussians) ---
    sift = cv2.SIFT_create()
    keypoints_sift, _ = sift.detectAndCompute(preprocessed, None)
    
    # the DRAW_RICH_KEYPOINTS flag is crucial: it draws rings showing the actual SIZE of the blob!
    img_sift = cv2.drawKeypoints(base_color_img, keypoints_sift, None, 
                                 flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS, 
                                 color=(0, 255, 0)) # Green circles
    
    # --- 2. ORB Blob Detection ---
    # We cap ORB at 500 features so the screen doesn't get flooded
    orb = cv2.ORB_create(nfeatures=500)
    keypoints_orb, _ = orb.detectAndCompute(preprocessed, None)
    
    img_orb = cv2.drawKeypoints(base_color_img, keypoints_orb, None, 
                                flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS, 
                                color=(255, 0, 0)) # Blue circles
                                
    # --- Plotting ---
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    axes[0].imshow(preprocessed, cmap='gray')
    axes[0].set_title("Input (Green + CLAHE)")
    axes[0].axis('off')
    
    axes[1].imshow(cv2.cvtColor(img_sift, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"SIFT Blobs (Found: {len(keypoints_sift)})")
    axes[1].axis('off')
    
    axes[2].imshow(cv2.cvtColor(img_orb, cv2.COLOR_BGR2RGB))
    axes[2].set_title(f"ORB Blobs (Found: {len(keypoints_orb)})")
    axes[2].axis('off')
    
    plt.subplots_adjust(wspace=0.1)
    plt.show()

# Apply Blob Detection to ALL 15 images in sequence.
for count, img_path in enumerate(all_15_image_paths, 1):
    print(f"[{count}/15] Applying Blob Detection on: {os.path.basename(img_path)}")
    test_blob_detection(img_path)
